We need to sign up for [an NGC account here](https://ngc.nvidia.com/signin). Once signed in you will be able to view the [NVIDIA NeMo Microservices Helm chart page](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/nemo-microservices/helm-charts/nemo-microservices-helm-chart). The NeMo microservices helm chart bundles the individual Helm charts of various NeMo microservices (incl. NeMo Data Store, Entity Store, Customizer, and more) so that we can easily deploy the full suite of NeMo microservices.

To fetch the Helm chart we need to grab our NGC API key. To do so, we navigate to **Setup > Generate API Key >** and for **Included Services** _ensure_ you check **"NGC Catalog"**. Take the generated API key and enter it below.

In [ ]:
from getpass import getpass

NGC_API_KEY = getpass("Enter your NGC API key: ")

We then fetch the Helm chart like so (note, the username of `'$oauthtoken'` _must not_ be changed):

In [ ]:
!helm fetch https://helm.ngc.nvidia.com/nvidia/nemo-microservices/charts/nemo-microservices-helm-chart-25.4.0.tgz \
    --username='$oauthtoken' \
    --password={NGC_API_KEY}

There are a few values we add to our Helm charts which we use a `values.yaml` file for - this file will overwrite or create these values in the underlying Helm charts that we just downloaded during their deployment.

We will set:

* `ngcAPIKey` which is simply our authentication method, this will be used by various microservice components to pull data from NGC.
* Which base model we'd like to fine-tune in NeMo Customizer via the `customizer.customizerConfig` values.
* A custom deployment storage definition inside `deployments.defaultStorageClass`.

In [ ]:
values = f"""
ngcAPIKey: {NGC_API_KEY}

customizer:
  customizerConfig:
    models:
      meta/llama-3.2-1b-instruct:
        enabled: true
        model_path: llama-3_2-1b-instruct
        max_seq_len: 4096
        num_parameters: 1000000000
        training_options:
          - training_type: sft
            finetuning_type: lora
            num_gpus: 1

deployments:
  defaultStorageClass: nfs-client
"""

with open("values.yaml", "w") as f:
    f.write(values.strip())

We use the following interchangeable values for our Helm _release name_, and Kubernetes _namespace_:

In [ ]:
NAMESPACE = "demo"
RELEASE_NAME = "training-demo"

Now we create the kubernetes namespace where our Helm chart will be used to setup our deployment.

In [ ]:
!kubectl create namespace {NAMESPACE}

Before deploying, we setup a rule that will automatically detect storage class fields and replace any empty fields with the `nfs-client` storage class - this will allow our Customizer to save model weights during training.

In [ ]:
assign_storageclass = """
apiVersion: kyverno.io/v1
kind: ClusterPolicy
metadata:
  name: add-default-storageclass
spec:
  rules:
    - name: add-storageclass-if-non-existent
      match:
        resources:
          kinds:
            - PersistentVolumeClaim
      mutate:
        patchStrategicMerge:
          spec:
            storageClassName: nfs-client
"""

with open("assign_storageclass.yaml", "w") as f:
    f.write(assign_storageclass.strip())

Download and install `kyverno` (which manages the detect-and-replace logic defined above).

In [ ]:
!kubectl create -f https://github.com/kyverno/kyverno/releases/download/v1.14.1/install.yaml

Then `apply` our storage class replacement logic within our cluster like so:

In [ ]:
!kubectl apply -f assign-storageclass.yaml -n {NAMESPACE} --validate=false

### Deployment

Now we're ready to push ahead with the deployment. We install the Helm chart into our Kubernetes cluster inside the `training-demo` namespace:

In [ ]:
!helm --namespace {NAMESPACE} \
    install {RELEASE_NAME} nemo-microservices-helm-chart-25.4.0.tgz \
    -f values.yaml

After installation we should see something like this:

```
NAME: training-demo
LAST DEPLOYED: Fri May 16 18:01:49 2025
NAMESPACE: demo
STATUS: deployed
REVISION: 1
```

Now we can check on the deployment in kubernetes:

In [ ]:
!kubectl get all -n {NAMESPACE}

You should see everything running. However, when running this for the first time you may see `pod/training-demo-nemo-operator-controller-manage` is _not_ running and stuck in a `CrashLoopBackOff`. If so, it is likely due to a missing scheduler dependency called [Volcano](https://github.com/volcano-sh/volcano). We must install this into our deployment separately like so:

In [ ]:
!kubectl apply -f https://raw.githubusercontent.com/volcano-sh/volcano/release-1.7/installer/volcano-development.yaml

Once Volcano is installed we delete the crashing pod:

!kubectl delete pod training-demo-nemo-operator-controller-manager-xyz -n {NAMESPACE}

Now when checking our deployment again, the `training-demo-nemo-operator-controller-manager` pod should be running:

!kubectl get pod -n {NAMESPACE}

### Setting NVCR Secret

The final step that we need is to set the `nvcrimagepullsecret` value - this is used by our training jobs to pull the base model images from NVIDIA's container registry (NVCR). First, we delete any preset secret value:

In [ ]:
!kubectl delete secret nvcrimagepullsecret -n {NAMESPACE} || true

Then create our secret using our login credentials, ie our NGC API key. As before, we _do not_ modify the `$oauthtoken` value.

In [ ]:
!kubectl --namespace {NAMESPACE} \
    create secret docker-registry nvcrimagepullsecret \
    --docker-server=nvcr.io \
    --docker-username='$oauthtoken' \
    --docker-password={NGC_API_KEY}

You can confirm that the secret has been updated by getting the secret record and checking the `AGE`:

In [ ]:
!kubectl get secret nvcrimagepullsecret -n {NAMESPACE}

With that, our NeMo Microservices are up and running!